# GRS-34806 MGI Project
## 1) Download the data
This template prepares the UCM data (both mono and multi-labels) inside data/raw in the project workspace. 

Write the notebook in such a way that it fully runs from start to end without further intervention  
(i.e. do not change the directory structure manually in the mean time).

Good luck!

In [1]:
from pathlib import Path
import shutil
import subprocess
import tempfile
import zipfile
import pandas as pd

# Get paths to important directories and files
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

IMAGES_DIR = RAW_DIR / "Images"
LABELS_FILE = RAW_DIR / "LandUse_Multilabeled.txt"

# Download and extract the dataset if not already present
if not IMAGES_DIR.exists() or not LABELS_FILE.exists():   
    with tempfile.TemporaryDirectory() as tmp:
        repo = Path(tmp) / "ucmdata"
        # Clone repo
        subprocess.run(["git", "clone", "--depth", "1", "https://git.wur.nl/lobry001/ucmdata.git", str(repo)], check=True)
        # Copy labels file
        shutil.copy2(repo / "LandUse_Multilabeled.txt", LABELS_FILE)
        # Unzip images and move to target directory
        with zipfile.ZipFile(repo / "UCMerced_LandUse.zip") as z:
            z.extractall(tmp)
        shutil.move(str(Path(tmp) / "UCMerced_LandUse" / "Images"), IMAGES_DIR)
        shutil.rmtree(repo)

UCM_images_path = str(IMAGES_DIR)
Multilabels_path = str(LABELS_FILE)

## 2) Data Preparation

In [2]:
# Get the multilabels data as a DataFrame
df = pd.read_csv(LABELS_FILE, sep=r"\s+", header=0)

# get path for images from 'IMAGE\LABEL' column
parts = df["IMAGE\\LABEL"].str.extract(r"^([^\d]+)(\d+)$")
df["folder"] = parts[0]
df["number"] = parts[1]

df["path"] = (
    str(IMAGES_DIR) + "\\" + df["folder"] + "\\" + df["folder"] + df["number"] + ".tif"
)

print(df.head())

      IMAGE\LABEL  airplane  bare-soil  buildings  cars  chaparral  court  \
0  agricultural00         0          0          0     0          0      0   
1  agricultural01         0          0          0     0          0      0   
2  agricultural02         0          0          0     0          0      0   
3  agricultural03         0          0          0     0          0      0   
4  agricultural04         0          0          0     0          0      0   

   dock  field  grass  ...  pavement  sand  sea  ship  tanks  trees  water  \
0     0      1      0  ...         0     0    0     0      0      1      0   
1     0      1      0  ...         0     0    0     0      0      0      0   
2     0      1      0  ...         0     0    0     0      0      0      0   
3     0      1      0  ...         0     0    0     0      0      0      0   
4     0      0      0  ...         0     0    0     0      0      1      0   

         folder number                                              

## 3) Train model

In [3]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.model_selection import train_test_split

In [4]:
# Get class columns
class_names = df.columns[1:-3]

num_classes = len(class_names)

print("Classes:", list(class_names))

Classes: ['airplane', 'bare-soil', 'buildings', 'cars', 'chaparral', 'court', 'dock', 'field', 'grass', 'mobile-home', 'pavement', 'sand', 'sea', 'ship', 'tanks', 'trees', 'water']


In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [6]:
labels = train_df[class_names].values

# Count positives per class
pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

# Avoid division by zero
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32)

print("pos_weight:", pos_weight)

pos_weight: tensor([19.2410,  1.9066,  1.9787,  1.3830, 18.0909, 19.4878, 20.0000, 19.4878,
         1.1734, 21.4000,  0.6123,  6.1186, 20.0000, 19.4878, 19.4878,  1.0766,
         9.2439])


In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # handles inconsistent sizes
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

In [8]:
class MultiLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row[self.class_names].values.astype(np.float32))

        return image, label

In [9]:
train_dataset = MultiLabelDataset(train_df, class_names, transform)
test_dataset  = MultiLabelDataset(test_df, class_names, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

model = models.resnet50(pretrained=True)

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

Using device: cuda


c:\Users\tomer\anaconda3\envs\Deep-learning-land-use-classification\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\tomer\anaconda3\envs\Deep-learning-land-use-classification\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [12]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [13]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [14]:
epochs = 5

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_loss  = evaluate(model, test_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss:  {test_loss:.4f}")

Epoch 1/5
Train Loss: 0.5303
Test Loss:  0.2414
Epoch 2/5
Train Loss: 0.2042
Test Loss:  0.1594
Epoch 3/5
Train Loss: 0.1408
Test Loss:  0.1452
Epoch 4/5
Train Loss: 0.1117
Test Loss:  0.1428
Epoch 5/5
Train Loss: 0.0872
Test Loss:  0.1577


In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score

def compute_metrics(model, loader, class_names, threshold=0.5):
    model.eval()

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs)

            preds = (probs > threshold).int().cpu().numpy()

            all_preds.append(preds)
            all_labels.append(labels.numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # Per-class metrics
    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall    = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1        = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # Macro averages (treat all classes equally)
    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro    = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro        = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return precision, recall, f1, precision_macro, recall_macro, f1_macro

In [18]:
precision, recall, f1, p_macro, r_macro, f1_macro = compute_metrics(
    model,
    test_loader,
    class_names,
    threshold=0.5
)

for i, cls in enumerate(class_names):
    print(f"{cls:15s} | Precision: {precision[i]:.3f} | Recall: {recall[i]:.3f} | F1: {f1[i]:.3f}")

print("\n--- Macro Averages ---")
print(f"Precision: {p_macro:.3f}")
print(f"Recall:    {r_macro:.3f}")
print(f"F1 Score:  {f1_macro:.3f}")

airplane        | Precision: 1.000 | Recall: 1.000 | F1: 1.000
bare-soil       | Precision: 0.726 | Recall: 0.850 | F1: 0.783
buildings       | Precision: 0.832 | Recall: 0.976 | F1: 0.899
cars            | Precision: 0.871 | Recall: 0.856 | F1: 0.864
chaparral       | Precision: 0.871 | Recall: 1.000 | F1: 0.931
court           | Precision: 0.917 | Recall: 0.957 | F1: 0.936
dock            | Precision: 1.000 | Recall: 1.000 | F1: 1.000
field           | Precision: 1.000 | Recall: 1.000 | F1: 1.000
grass           | Precision: 0.919 | Recall: 0.896 | F1: 0.907
mobile-home     | Precision: 0.964 | Recall: 1.000 | F1: 0.982
pavement        | Precision: 0.938 | Recall: 0.930 | F1: 0.934
sand            | Precision: 0.895 | Recall: 0.879 | F1: 0.887
sea             | Precision: 1.000 | Recall: 1.000 | F1: 1.000
ship            | Precision: 1.000 | Recall: 1.000 | F1: 1.000
tanks           | Precision: 1.000 | Recall: 0.889 | F1: 0.941
trees           | Precision: 0.906 | Recall: 0.870 | F1